In [ ]:
import os
import numpy as np
os.environ.pop("DEBUG", None)
# os.environ["BEAM"] = "2"
from tinygrad import Tensor, Device, TinyJit
from tinygrad.nn.optim import AdamW
from tinygrad.nn.state import get_parameters
from engine.game import GameBatch, GameResult, Color, MAX_MOVES
from models.v0 import Model, Config, init_weights
from helpers.evaluator import Evaluator, Encoding
from helpers.replay_buffer import ReplayBuffer

GAME_BATCH_SIZE = 64
MODEL_BATCH_SIZE = 64
BUFFER_CAPACITY = 200_000

rng = np.random.default_rng(42)	

In [ ]:
batch = GameBatch(GAME_BATCH_SIZE)
config = Config()
model = Model(config)
init_weights(model, config)

evaluator = Evaluator(model, MODEL_BATCH_SIZE)
replay_buffer = ReplayBuffer(BUFFER_CAPACITY)

In [ ]:
def sigma(q, max_N):
	return (50.0 + max_N) * q

# IDEALLY, USE TINYGRAD GPU INSTEAD!
def np_softmax(x: np.ndarray, axis=-1):
	x_max = np.max(x, axis=axis, keepdims=True)
	exp_x = np.exp(x - x_max)
	return exp_x / np.sum(exp_x, axis=axis, keepdims=True)

class Node:
	__slots__ = (
		"n_moves", "terminal", "expanded", "children",
		"value", "p_logits", "N", "W"
	)

	def __init__(self, 
		n_moves: int, 
		terminal: bool = False, 
		value: float = 0.0
	):
		self.n_moves = n_moves
		self.terminal = terminal
		self.value = value
		self.expanded: bool = False
		self.p_logits: np.ndarray
		self.N: np.ndarray = None
		self.W: np.ndarray = None
		self.children: list[Node] = None
	
	def expand(self, priors):
		self.p_logits = priors
		self.N = np.zeros(self.n_moves, dtype=np.float32)
		self.W = np.zeros(self.n_moves, dtype=np.float32)
		self.children = [None for _ in range(self.n_moves)]
		self.expanded = True
	
	def Q(self, a):
		return self.W[a] / self.N[a]
	# definitely needs to be improved with numpy / tinygrad GPU vectorization
	def v_mix(self):
		visited = [a for a in range(self.n_moves) if self.N[a] > 0]
		if not visited: return self.value

		policy = np_softmax(self.p_logits)
		active_policy_sum = sum(policy[a] for a in visited)
		policy_mean_value = sum(
			policy[a] * self.Q(a) for a in visited
		) / active_policy_sum
		node_N = self.N.sum()
		return (self.value + node_N * policy_mean_value) / (1 + node_N)
	# definitely needs vectorization
	def completed_Q(self):
		_v_mix = self.v_mix()
		return [
			self.Q(a) if self.N[a] > 0 else _v_mix
			for a in range(self.n_moves)
		]
	def improved_policy(self):
		_completed_Q = self.completed_Q()
		max_N = self.N.max() if self.expanded else 0
		return np_softmax([
			self.p_logits[a] + sigma(_completed_Q[a], max_N)
			for a in range(self.n_moves)
		])
	def select(self):
		_improved_policy = self.improved_policy()
		node_N = self.N.sum()
		return max(
			range(self.n_moves),
			key=lambda a: _improved_policy[a] - self.N[a] / (1 + node_N)
		)

def run_gumbel(
	game_batch: GameBatch,
	evaluator: Evaluator,
	rng: np.random.Generator,
	n_sims: int, 
	k: int
):
	B = len(game_batch)
	active = game_batch.active
	Encoding(game_batch.get_encoding())